# CAP1 — Gestão de Crises com Sistema Preditivo em IA
Execute em ordem no Colab. Registre apenas métricas realmente produzidas.

In [ ]:
!pip -q install numpy pandas scikit-learn

In [ ]:
import numpy as np, pandas as pd, json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
np.random.seed(42); n=5000
df=pd.DataFrame({"idade_media":np.random.normal(55,12,n).clip(18,90),"carga_sistema":np.random.uniform(30,100,n),"tempo_espera":np.random.normal(45,20,n).clip(5,180),"fc_media":np.random.normal(85,15,n).clip(50,150),"spo2_media":np.random.normal(95,3,n).clip(70,100),"pa_sistolica":np.random.normal(125,20,n).clip(80,200),"temperatura":np.random.normal(24,5,n).clip(10,40),"densidade_eventos":np.random.poisson(3,n),"eventos_24h":np.random.poisson(2,n),"disponibilidade_recursos":np.random.uniform(20,100,n)})
s=( (df.carga_sistema>80).astype(int)+(df.tempo_espera>60).astype(int)+(df.spo2_media<92).astype(int)+(df.fc_media>110).astype(int)+(df.eventos_24h>4).astype(int)+(df.disponibilidade_recursos<40).astype(int)); df["pico_risco"]=(s>=2).astype(int)
features=["idade_media","carga_sistema","tempo_espera","fc_media","spo2_media","pa_sistolica","temperatura","densidade_eventos","eventos_24h","disponibilidade_recursos"]; X,y=df[features],df.pico_risco
Xtr,Xtmp,ytr,ytmp=train_test_split(X,y,test_size=.30,random_state=42,stratify=y); Xv,Xte,yv,yte=train_test_split(Xtmp,ytmp,test_size=.50,random_state=42,stratify=ytmp)
m=RandomForestClassifier(n_estimators=300,random_state=42,class_weight="balanced",n_jobs=-1).fit(Xtr,ytr)
def metrics(a,b): return {"accuracy":accuracy_score(a,b),"precision":precision_score(a,b,zero_division=0),"recall":recall_score(a,b,zero_division=0),"f1":f1_score(a,b,zero_division=0),"confusion_matrix":confusion_matrix(a,b).tolist(),"report":classification_report(a,b,output_dict=True,zero_division=0)}
results={"split":{"treino":len(Xtr),"validacao":len(Xv),"teste":len(Xte)},"distribuicao_classes":y.value_counts().sort_index().to_dict(),"validacao":metrics(yv,m.predict(Xv)),"teste":metrics(yte,m.predict(Xte))}
print(json.dumps(results,indent=2))


Limitações: dados sintéticos; score não é probabilidade clínica calibrada. Temperatura é preservada conforme enunciado e não participa do rótulo. Não usar em decisões reais.